# 07. The counter-based search index

This notebook shows our improvement to QPASE (`docs/counter-index-spec.md`, slides 14 to 20 of
`QPASE_Paper_Presentation_counter_index-2.pdf`). It stores the same seven files as
`03_outsource_retrieve.ipynb`, on the same kind of servers, so the two notebooks can be read side by side.

In the paper's scheme a search sends $dsk = HKDF(K_u, w)$ to every server, and each server tries $dsk$ on
every stored item. A server that keeps $dsk$ can test every later upload with it. Here the user never sends
$K_w$ (the old $dsk$). The $j$-th file under $w$ gets the label $label(w, j) = PRF(K_w, j)$, and a server
stores one row $label \to entry$ per file. To search, the user sends the labels for $j = 1, \dots, c_w$, and
each server does $c_w$ dictionary lookups. A later upload gets the label for $j = c_w + 1$, which the server
cannot compute, so old searches do not find it (forward privacy).

In [1]:
import dataclasses
import time

from ris_sem_project import protocol
from ris_sem_project.counter_index import open_counter_table, open_entry
from ris_sem_project.display import payload_size, peek, table
from ris_sem_project.index import ScanOutcome, scan_item
from ris_sem_project.params import PARAM_I, Lattice, setup
from ris_sem_project.parties.counter_index_server import CounterIndexServer, StoreOutcome
from ris_sem_project.parties.counter_index_user import CounterIndexUser

PASSWORD = "correct horse battery staple"


def flip(data):
    middle = len(data) // 2
    return data[:middle] + bytes([data[middle] ^ 1]) + data[middle + 1 :]

## Setup, Register, and Login with step L4

$N = 3$ servers, threshold $t = 2$, all of type `CounterIndexServer`, which also runs every phase of the
paper. Register and Login L1 to L3 are unchanged. In the new step L4 the user asks the contacted servers for
the sealed counter table. Alice has never uploaded one, so both servers return none and her table is
empty at version 0. Her session also holds $k_{ptr}$ and $k_{ctr}$, the first two 32-byte blocks of one
HKDF output $HKDF(K_u, \texttt{"QPASE/ci/keys"})$.

In [2]:
pp = setup(PARAM_I, N=3, t=2, mu=20)
lattice = Lattice.from_pp(pp)
servers = protocol.setup_servers(pp, server_type=CounterIndexServer)
alice = CounterIndexUser(pp, "alice", PASSWORD)
registered = protocol.register(alice, servers)
session = protocol.ci_login(alice, servers, (1, 2))
keys = alice.index_keys
print(f"Register on S_1, S_2, S_3: pk_u = {peek(registered.key_message.user_pk)}")
print(f"Login L1-L3 with S_1, S_2: K_u = {peek(session.login.k_u)}, same as Register: {session.login.k_u == registered.k_u}")
print(f"L4, counter table returned by each server: {', '.join(f'S_{i}: {upload}' for i, upload in session.uploads.items())}")
print(f"counter table at U: version {session.table.version}, counts {dict(session.table.counts)}")
print(f"index keys: k_ptr {peek(keys.k_ptr)}, k_ctr {peek(keys.k_ctr)}")

Register on S_1, S_2, S_3: pk_u = 2ff7…f840 (1312 B)
Login L1-L3 with S_1, S_2: K_u = 440b…a4a8 (1792 B), same as Register: True
L4, counter table returned by each server: S_1: None, S_2: None
counter table at U: version 0, counts {}
index keys: k_ptr d4e3…c113 (32 B), k_ctr e1b8…2022 (32 B)


## The files

The seven files of notebook 03, each with an item id $i$, one keyword $w$ and the server it goes to.

In [3]:
CORPUS = [
    (1, "medical", 1, "Blood test results, Riverside Lab, 14 Sep 2026. Haemoglobin 14.1 g/dL, LDL cholesterol 2.9 mmol/L, HbA1c 5.2%. All within range, no follow-up needed."),
    (2, "invoice", 2, "Invoice 2026-0417, Northside Plumbing. Replaced kitchen tap and fixed the leak under the sink: 2.5 h labour plus parts, EUR 186.40, due 30 Oct 2026."),
    (3, "medical", 3, "Physiotherapy referral from Dr. Okafor. Lumbar MRI shows a mild disc bulge at L4-L5 with no nerve compression. Six sessions, twice a week."),
    (4, "travel", 1, "Flight booking KX42QP: Lisbon LIS to Berlin BER on 3 Nov 2026, departs 09:40, seat 23A, one checked bag."),
    (5, "meeting", 2, "Project sync, 21 Oct. Priya ships the export fix on Friday. Tom drafts the Q4 budget. Next review 4 Nov in room 3.12."),
    (6, "medical", 2, "Vaccination record: influenza 2026/27 dose given 2 Oct 2026, batch FL2611, left arm. Tetanus booster due 2031."),
    (7, "travel", 3, "Hotel Spreeblick confirmation 88231: 3-7 Nov 2026, 4 nights, double room, breakfast included, check-in from 15:00."),
]
DOCUMENTS = {item_id: text.encode() for item_id, _, _, text in CORPUS}
print(table(("item i", "keyword w", "server", "|d|", "file d (start)"), [
    (item_id, keyword, f"S_{server}", f"{len(DOCUMENTS[item_id])} B", text[:48] + "...")
    for item_id, keyword, server, text in CORPUS
]))

item i  keyword w  server  |d|    file d (start)
------  ---------  ------  -----  ---------------------------------------------------
1       medical    S_1     149 B  Blood test results, Riverside Lab, 14 Sep 2026. ...
2       invoice    S_2     148 B  Invoice 2026-0417, Northside Plumbing. Replaced ...
3       medical    S_3     138 B  Physiotherapy referral from Dr. Okafor. Lumbar M...
4       travel     S_1     104 B  Flight booking KX42QP: Lisbon LIS to Berlin BER ...
5       meeting    S_2     117 B  Project sync, 21 Oct. Priya ships the export fix...
6       medical    S_2     110 B  Vaccination record: influenza 2026/27 dose given...
7       travel     S_3     114 B  Hotel Spreeblick confirmation 88231: 3-7 Nov 202...


## Outsource item 1, step by step

Steps 1 and 2. The user sets $j = c_w + 1$ and computes $label(w, j)$, the first 32 bytes of
$PRF(K_w, j)$: one lattice PRF chain. It encrypts $C = Enc(K_u, d)$ as in the paper and seals the entry
$Enc(k_{ptr}, (label, i, SHA3\text{-}256(C)))$. It signs $(label, entry, i, C)$ and sends all of it to the
server it chose. $K_w$ is computed on the device and does not leave it. The cell opens the entry with Alice's
own $k_{ptr}$ to print what it holds.

In [4]:
item_id, keyword, server_id, _ = CORPUS[0]
j = alice.counter_table.count(keyword) + 1
request = alice.indexed_outsource(keyword, DOCUMENTS[item_id], item_id)
inside = open_entry(keys.k_ptr, request.label, request.entry)
print(f"w = {keyword!r}, c_w = {j - 1}, so j = {j}")
print(f"K_w = HKDF(K_u, w): {peek(alice.search_key(keyword))}, stays on the device")
print(f"label(w, {j}):       {peek(request.label)}")
print(f"C = Enc(K_u, d):    {peek(request.data_ct)}")
print(f"entry:              {peek(request.entry)}, holds i = {inside.item_id} and SHA3-256(C) = {peek(inside.file_hash)}")
print(f"signature:          {peek(request.signature)}")
print(f"IndexedOutsourceRequest to S_{server_id}: label {len(request.label)} B + entry {len(request.entry)} B + C {len(request.data_ct)} B + signature {len(request.signature)} B = {payload_size(request)} B with the user id")

w = 'medical', c_w = 0, so j = 1
K_w = HKDF(K_u, w): 897c…3a90 (1792 B), stays on the device
label(w, 1):       9c36…89f7 (32 B)
C = Enc(K_u, d):    4152…a715 (177 B)
entry:              8513…291c (108 B), holds i = 1 and SHA3-256(C) = 8c14…4837 (32 B)
signature:          62ff…2a38 (2420 B)
IndexedOutsourceRequest to S_1: label 32 B + entry 108 B + C 177 B + signature 2420 B = 2742 B with the user id


## Step 3 at $S_1$, step 4 at Alice

$S_1$ verifies the signature under $pk_u$, checks that the label is new and that Alice has a free row, and
stores the row and $C$. The cell first sends a copy with one bit of $C$ flipped in transit, which $S_1$
rejects. Only after $S_1$ stored the row does Alice set $c_w = j$. She then seals the counter table under
$k_{ctr}$, padded to a multiple of 1024 bytes, signs it with the next version, and uploads it to all $N$ servers.

In [5]:
server = servers[server_id]
forged_outcome = server.indexed_outsource_store(dataclasses.replace(request, data_ct=flip(request.data_ct)))
outcome = server.indexed_outsource_store(request)
upload = alice.confirm_outsource(keyword)
table_accepted = {i: s.store_counter_table(upload) for i, s in servers.items()}
print(f"S_{server_id}, C tampered in transit: {forged_outcome.name}")
print(f"S_{server_id}, the real request: {outcome.name}")
print(f"step 4: c_w = {alice.counter_table.count(keyword)}, counter table version {upload.version}, sealed {len(upload.sealed)} B")
print(f"accepted by: {', '.join(f'S_{i}' for i, ok in table_accepted.items() if ok)}")
print(f"CounterTableUpload to each server: {payload_size(upload)} B with the user id")

S_1, C tampered in transit: BAD_SIGNATURE
S_1, the real request: STORED
step 4: c_w = 1, counter table version 1, sealed 1072 B
accepted by: S_1, S_2, S_3
CounterTableUpload to each server: 3497 B with the user id


## Outsource the other files

The same steps for items 2 to 7, each to its own server, through `protocol.ci_outsource`. Each upload
raises the counter table's version by one.

In [6]:
outsourced = {item_id: request}
versions = {item_id: upload.version}
for item_id, keyword, server_id, _ in CORPUS[1:]:
    run = protocol.ci_outsource(alice, servers, server_id, keyword, DOCUMENTS[item_id], item_id)
    outsourced[item_id] = run.request
    versions[item_id] = run.upload.version
print(table(("item i", "keyword w", "server", "label", "|entry|", "|C|", "table version"), [
    (item_id, keyword, f"S_{server_id}", peek(outsourced[item_id].label), len(outsourced[item_id].entry), len(outsourced[item_id].data_ct), versions[item_id])
    for item_id, keyword, server_id, _ in CORPUS
]))

item i  keyword w  server  label             |entry|  |C|  table version
------  ---------  ------  ----------------  -------  ---  -------------
1       medical    S_1     9c36…89f7 (32 B)  108      177  1
2       invoice    S_2     d296…667e (32 B)  108      176  2
3       medical    S_3     420b…74d3 (32 B)  108      166  3
4       travel     S_1     4509…9407 (32 B)  108      132  4
5       meeting    S_2     cf04…5708 (32 B)  108      145  5
6       medical    S_2     7098…393c (32 B)  108      138  6
7       travel     S_3     35c8…3ac8 (32 B)  108      142  7


## What each server holds

Each server keeps random-looking rows, the files $C$ under their item ids, and the sealed counter table.
The cell searches every byte string a server holds for each keyword and finds none.

In [7]:
for i, s in servers.items():
    held = s.counter_tables["alice"]
    print(f"S_{i}: counter table version {held.version}, sealed {len(held.sealed)} B; files {[(item, len(c)) for item, c in s.files['alice'].items()]} as (i, |C|)")
    print(table(("label", "entry"), [(peek(label), peek(entry)) for label, entry in s.index["alice"].items()]))
    print()
state = {
    i: [b for row in s.index["alice"].items() for b in row] + list(s.files["alice"].values()) + [s.counter_tables["alice"].sealed]
    for i, s in servers.items()
}
keyword_found = {w: [f"S_{i}" for i, blobs in state.items() if any(w.encode() in b for b in blobs)] for w in sorted({w for _, w, _, _ in CORPUS})}
print(f"servers whose stored bytes contain the keyword: {keyword_found}")

S_1: counter table version 7, sealed 1072 B; files [(1, 177), (4, 132)] as (i, |C|)
label             entry
----------------  -----------------
9c36…89f7 (32 B)  8513…291c (108 B)
4509…9407 (32 B)  2cb9…4904 (108 B)

S_2: counter table version 7, sealed 1072 B; files [(2, 176), (5, 145), (6, 138)] as (i, |C|)
label             entry
----------------  -----------------
d296…667e (32 B)  b98a…da5d (108 B)
cf04…5708 (32 B)  7237…3e17 (108 B)
7098…393c (32 B)  62b6…7d45 (108 B)

S_3: counter table version 7, sealed 1072 B; files [(3, 166), (7, 142)] as (i, |C|)
label             entry
----------------  -----------------
420b…74d3 (32 B)  1d99…86f3 (108 B)
35c8…3ac8 (32 B)  37ec…477f (108 B)

servers whose stored bytes contain the keyword: {'invoice': [], 'medical': [], 'meeting': [], 'travel': []}


## The counter table

Alice holds the table in memory. Every server holds the same signed upload, sealed under $k_{ctr}$, which
only Alice can open. The servers learn its size in 1024-byte blocks and its version.

In [8]:
held = servers[1].counter_tables["alice"]
print(f"U holds: version {alice.counter_table.version}, counts {dict(alice.counter_table.counts)}")
print(f"each server holds: version {held.version}, sealed {peek(held.sealed)}, signature {peek(held.signature)}")
print(f"every server holds the same upload: {len({s.counter_tables['alice'] for s in servers.values()}) == 1}")
print(f"Dec(k_ctr, sealed) at U gives back U's table: {open_counter_table(keys.k_ctr, held.sealed) == alice.counter_table}")

U holds: version 7, counts {'medical': 3, 'invoice': 1, 'travel': 2, 'meeting': 1}
each server holds: version 7, sealed c171…2d0c (1072 B), signature 50d9…3945 (2420 B)
every server holds the same upload: True
Dec(k_ctr, sealed) at U gives back U's table: True


## R1: Alice searches for `medical`

$c_w = 3$, so Alice computes $label(w, 1), label(w, 2), label(w, 3)$, signs them, and sends them to all $N$
servers, since she does not know which server holds which row.

In [9]:
QUERY = "medical"
expected_ids = [item_id for item_id, keyword, _, _ in CORPUS if keyword == QUERY]
request = alice.label_search_request(QUERY)
print(f"w = {QUERY!r}, c_w = {alice.counter_table.count(QUERY)}")
for j, label in enumerate(request.labels, start=1):
    print(f"label(w, {j}) = {peek(label)}")
print(f"signature: {peek(request.signature)}")
print(f"U -> each of the {len(servers)} servers: LabelSearchRequest, {payload_size(request)} B")

w = 'medical', c_w = 3
label(w, 1) = 9c36…89f7 (32 B)
label(w, 2) = 420b…74d3 (32 B)
label(w, 3) = 7098…393c (32 B)
signature: 4a80…2e45 (2420 B)
U -> each of the 3 servers: LabelSearchRequest, 2521 B


## R2 on every server

Each server verifies the signature and looks each label up in Alice's dictionary. It runs no PRF and
decrypts nothing, and its work does not depend on how many rows it stores.

In [10]:
start = time.perf_counter()
searches = {i: s.label_search(request) for i, s in servers.items()}
r2_seconds = time.perf_counter() - start
print(table(("server", "rows held", "lookups", "hits"), [
    (f"S_{i}", len(servers[i].index["alice"]), search.lookups, ", ".join(f"label(w, {request.labels.index(label) + 1})" for label, _ in search.response.hits) or "none")
    for i, search in searches.items()
]))
print(f"R2 on all {len(servers)} servers: {r2_seconds * 1e3:.0f} ms")

server  rows held  lookups  hits
------  ---------  -------  -----------
S_1     2          3        label(w, 1)
S_2     3          3        label(w, 3)
S_3     2          3        label(w, 2)
R2 on all 3 servers: 27 ms


## R3 at Alice

Alice keeps a returned row only if its label is one she asked for and its entry opens under $k_{ptr}$
with that same label inside. Each entry gives the item id and the hash the file must have.

In [11]:
hits = alice.open_hits(QUERY, [search.response for search in searches.values()])
print(table(("j", "from", "label", "i (in the entry)", "SHA3-256(C) (in the entry)"), [
    (hit.position, f"S_{hit.server_id}", peek(hit.label), hit.item_id, peek(hit.file_hash)) for hit in hits
]))

j  from  label             i (in the entry)  SHA3-256(C) (in the entry)
-  ----  ----------------  ----------------  --------------------------
1  S_1   9c36…89f7 (32 B)  1                 8c14…4837 (32 B)
2  S_3   420b…74d3 (32 B)  3                 c6a8…96fd (32 B)
3  S_2   7098…393c (32 B)  6                 2954…ee5c (32 B)


## R4 and R5: fetch the files

The server cannot follow an encrypted pointer, so Alice asks each server that returned a row for the files
by item id, in a second round trip. Here the server learns which files the search returned. Alice checks
$SHA3\text{-}256(C)$ against the entry and decrypts $d = Dec(K_u, C)$.

In [12]:
wanted = {}
for hit in hits:
    wanted.setdefault(hit.server_id, []).append(hit.item_id)
fetch_requests = {i: alice.fetch_request(sorted(ids)) for i, ids in sorted(wanted.items())}
fetch_responses = {i: servers[i].fetch_files(fetch) for i, fetch in fetch_requests.items()}
for i, fetch in fetch_requests.items():
    returned = [(item, len(c)) for item, c in fetch_responses[i].files]
    print(f"U -> S_{i}: FileFetchRequest for items {list(fetch.item_ids)}, {payload_size(fetch)} B; S_{i} -> U: {returned} as (i, |C|)")
files = alice.open_files(hits, list(fetch_responses.values()))
print()
for item_id, d in zip(sorted(hit.item_id for hit in hits), files):
    print(f"item {item_id}, hash matches the entry, d = Dec(K_u, C):")
    print(f"  {d.decode()}")

U -> S_1: FileFetchRequest for items [1], 2425 B; S_1 -> U: [(1, 177)] as (i, |C|)
U -> S_2: FileFetchRequest for items [6], 2425 B; S_2 -> U: [(6, 138)] as (i, |C|)
U -> S_3: FileFetchRequest for items [3], 2425 B; S_3 -> U: [(3, 166)] as (i, |C|)

item 1, hash matches the entry, d = Dec(K_u, C):
  Blood test results, Riverside Lab, 14 Sep 2026. Haemoglobin 14.1 g/dL, LDL cholesterol 2.9 mmol/L, HbA1c 5.2%. All within range, no follow-up needed.
item 3, hash matches the entry, d = Dec(K_u, C):
  Physiotherapy referral from Dr. Okafor. Lumbar MRI shows a mild disc bulge at L4-L5 with no nerve compression. Six sessions, twice a week.
item 6, hash matches the entry, d = Dec(K_u, C):
  Vaccination record: influenza 2026/27 dose given 2 Oct 2026, batch FL2611, left arm. Tetanus booster due 2031.


In [13]:
recovered = sorted(hit.item_id for hit in hits)
assert recovered == expected_ids, "exactly the files filed under the keyword are found"
assert files == [DOCUMENTS[i] for i in expected_ids], "each recovered d equals the original"
assert all(search.lookups == len(expected_ids) for search in searches.values()), "each server does c_w lookups"
print(f"{QUERY!r}: recovered items {recovered}, equal to the originals; lookups per server {[s.lookups for s in searches.values()]}")

'medical': recovered items [1, 3, 6], equal to the originals; lookups per server [3, 3, 3]


## Forward privacy

The servers keep Alice's search request. Alice then uploads a new `medical` file. Its label is
$label(w, 4)$, which is not in the kept request, and no server can compute it without $K_w$. Replaying the
kept request returns the old rows only. A new search by Alice sends four labels and finds all four files.

In [14]:
NEW_ID = 8
NEW_TEXT = b"Allergy test, 28 Oct 2026: mild reaction to birch pollen, none to penicillin. Antihistamine as needed."
kept = request
new_run = protocol.ci_outsource(alice, servers, 3, QUERY, NEW_TEXT, NEW_ID)
print(f"new medical file, item {NEW_ID} on S_3, label(w, {alice.counter_table.count(QUERY)}) = {peek(new_run.request.label)}")
print(f"new label among the kept request's labels: {new_run.request.label in kept.labels}")
replayed = {i: s.label_search(kept) for i, s in servers.items()}
replay_ids = sorted(hit.item_id for hit in alice.open_hits(QUERY, [search.response for search in replayed.values()]))
print(f"replaying the kept request: hits per server {[len(search.response.hits) for search in replayed.values()]}, items {replay_ids}")
fresh = protocol.ci_retrieve(alice, servers, QUERY)
fresh_ids = sorted(hit.item_id for hit in fresh.hits)
print(f"a new search by Alice: {len(fresh.labels)} labels, items {fresh_ids}")

new medical file, item 8 on S_3, label(w, 4) = 3b71…e68d (32 B)
new label among the kept request's labels: False
replaying the kept request: hits per server [1, 1, 1], items [1, 3, 6]
a new search by Alice: 4 labels, items [1, 3, 6, 8]


## The paper's scheme, for contrast

The same servers also run the paper's Outsource and Retrieve. A server keeps $dsk$ from one search and
tests an item uploaded afterwards with `index.scan_item`, the check it runs in R2. The later item matches.

In [15]:
protocol.outsource(alice, servers[1], QUERY, DOCUMENTS[1], 101)
paper_kept = protocol.retrieve(alice, servers, QUERY).request
paper_later = protocol.outsource(alice, servers[3], QUERY, NEW_TEXT, 108).request
scan = scan_item(lattice, registered.key_message.user_pk, paper_kept.search_keys[0], paper_later.item)
print(f"the servers kept dsk = {peek(paper_kept.search_keys[0])} from a search")
print(f"testing item {paper_later.item.item_id}, uploaded after that search, with the kept dsk: {scan.outcome.name}")

the servers kept dsk = 897c…3a90 (1792 B) from a search
testing item 108, uploaded after that search, with the kept dsk: MATCH


## Login from a new device

A new `CounterIndexUser` object knows only the identity and the password. It logs in with $S_2, S_3$ and
gets the counter table in L4, so it can search at once.

In [16]:
phone = CounterIndexUser(pp, "alice", PASSWORD)
phone_login = protocol.ci_login(phone, servers, (2, 3))
phone_search = protocol.ci_retrieve(phone, servers, QUERY)
print(f"L4 with S_2, S_3: versions returned {[upload.version for upload in phone_login.uploads.values()]}")
print(f"counter table on the new device: version {phone_login.table.version}, counts {dict(phone_login.table.counts)}")
print(f"same table as the first device: {phone_login.table == alice.counter_table}")
print(f"the new device searches {QUERY!r}: items {sorted(hit.item_id for hit in phone_search.hits)}")

L4 with S_2, S_3: versions returned [8, 8]
counter table on the new device: version 8, counts {'invoice': 1, 'medical': 4, 'meeting': 1, 'travel': 2}
same table as the first device: True
the new device searches 'medical': items [1, 3, 6, 8]


## Checks

In [17]:
assert forged_outcome is StoreOutcome.BAD_SIGNATURE and outcome is StoreOutcome.STORED, "step 3 stores only a correctly signed row"
assert all(table_accepted.values()), "every server accepts the first counter table"
assert all(not found for found in keyword_found.values()), "no keyword appears in server state"
assert new_run.request.label not in kept.labels, "the new row's label is not in the kept request"
assert replay_ids == expected_ids, "the kept request does not find the later upload"
assert fresh_ids == expected_ids + [NEW_ID], "a new search finds every file"
assert fresh.files == tuple(DOCUMENTS[i] for i in expected_ids) + (NEW_TEXT,), "every recovered d equals the original"
assert scan.outcome is ScanOutcome.MATCH, "in the paper's scheme a kept dsk matches a later upload"
assert phone_login.table == alice.counter_table, "the new device gets the same counters"
assert phone_search.files == fresh.files, "the new device finds the same files"
print("all counter index checks passed")

all counter index checks passed
